In [166]:
import pandas as pd
import numpy as np
import pickle

In [167]:
risk_model = pickle.load(open("../notebooks/risk_model.pkl","rb"))
risk_scaler = pickle.load(open("../notebooks/risk_scaler.pkl","rb"))
risk_reverse_mapping = pickle.load(open("../notebooks/risk_label_encoder.pkl","rb"))
validation_data = pd.read_csv("risk_validation.csv")
validation_data.to_csv("risk_validation.csv", index=False)


In [168]:
risk_x = validation_data.drop(
    columns=["Scenario", "Expected_Risk"]
)

risk_x.head()

,crowd_count,occupancy_pct,queue_length,avg_waiting_time_min,weather,weather_severity,crowd_behavior,medical_incidents,security_incidents,fire_alarm,...,bomb_threat,emergency_evacuation,volunteer_gap,camera_failure,network_failure,power_failure,road_traffic_index,visibility_km,Predicted_Risk,Result
0,1800,45,20,3,Sunny,0.0,Calm,0,0,0,...,0,0,2,0,0,0,25,10,Safe,PASS
1,5200,72,260,14,Cloudy,0.5,Normal,1,1,0,...,0,0,12,0,0,0,55,8,Dangerous,FAIL
2,9200,96,780,24,Rain,2.0,Aggressive,3,2,0,...,0,1,35,0,1,0,82,4,Dangerous,PASS
3,650,32,8,2,Sunny,0.0,Calm,0,0,0,...,0,0,1,0,0,0,18,10,Safe,PASS
4,28000,88,520,18,Cloudy,0.5,Normal,2,2,0,...,0,0,20,0,0,0,68,9,Dangerous,FAIL


In [169]:
risk_x = pd.get_dummies(risk_x)

training_columns = risk_model.feature_names_in_

risk_x = risk_x.reindex(
    columns=training_columns,
    fill_value=0
)

risk_x.head()

,crowd_count,occupancy_pct,queue_length,avg_waiting_time_min,weather_severity,medical_incidents,security_incidents,fire_alarm,stampede_alert,bomb_threat,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,1800,45,20,3,0.0,0,0,0,0,0,...,False,False,False,False,False,True,False,True,False,False
1,5200,72,260,14,0.5,1,1,0,0,0,...,True,False,False,False,False,False,False,False,True,False
2,9200,96,780,24,2.0,3,2,0,1,0,...,False,False,False,True,False,False,True,False,False,False
3,650,32,8,2,0.0,0,0,0,0,0,...,False,False,False,False,False,True,False,True,False,False
4,28000,88,520,18,0.5,2,2,0,0,0,...,True,False,False,False,False,False,False,False,True,False


In [170]:
numeric_cols = [
    "crowd_count",
    "occupancy_pct",
    "queue_length",
    "avg_waiting_time_min",
    "weather_severity",
    "medical_incidents",
    "security_incidents",
    "fire_alarm",
    "stampede_alert",
    "bomb_threat",
    "emergency_evacuation",
    "volunteer_gap",
    "camera_failure",
    "network_failure",
    "power_failure",
    "road_traffic_index",
    "visibility_km"
]

risk_x[numeric_cols] = risk_scaler.transform(
    risk_x[numeric_cols]
)

risk_x.head()

,crowd_count,occupancy_pct,queue_length,avg_waiting_time_min,weather_severity,medical_incidents,security_incidents,fire_alarm,stampede_alert,bomb_threat,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,-0.540248,-1.562046,-0.141978,-0.659214,-0.906447,-0.236235,-0.207684,-0.065716,-0.034662,-0.020004,...,False,False,False,False,False,True,False,True,False,False
1,-0.287253,-0.123925,0.187653,2.952451,-0.419660,1.244865,1.824453,-0.065716,-0.034662,-0.020004,...,True,False,False,False,False,False,False,False,True,False
2,0.010388,1.154403,0.901855,6.235784,1.040703,4.207064,3.856591,-0.065716,28.850188,-0.020004,...,False,False,False,True,False,False,True,False,False,False
3,-0.625820,-2.254474,-0.158460,-0.987548,-0.906447,-0.236235,-0.207684,-0.065716,-0.034662,-0.020004,...,False,False,False,False,False,True,False,True,False,False
4,1.409302,0.728294,0.544754,4.265784,-0.419660,2.725965,3.856591,-0.065716,-0.034662,-0.020004,...,True,False,False,False,False,False,False,False,True,False


In [171]:
risk_prediction = risk_model.predict(risk_x)

risk_prediction = pd.Series(risk_prediction).map(risk_reverse_mapping)

validation_data["Predicted_Risk"] = risk_prediction

validation_data.head()

,Scenario,crowd_count,occupancy_pct,queue_length,avg_waiting_time_min,weather,weather_severity,crowd_behavior,medical_incidents,security_incidents,...,emergency_evacuation,volunteer_gap,camera_failure,network_failure,power_failure,road_traffic_index,visibility_km,Expected_Risk,Predicted_Risk,Result
0,Shopping Mall - Normal Weekend,1800,45,20,3,Sunny,0.0,Calm,0,0,...,0,2,0,0,0,25,10,Safe,Safe,PASS
1,Airport Check-in Rush Hour,5200,72,260,14,Cloudy,0.5,Normal,1,1,...,0,12,0,0,0,55,8,Warning,Dangerous,FAIL
2,Railway Station Festival Rush,9200,96,780,24,Rain,2.0,Aggressive,3,2,...,1,35,0,1,0,82,4,Dangerous,Dangerous,PASS
3,Corporate Event,650,32,8,2,Sunny,0.0,Calm,0,0,...,0,1,0,0,0,18,10,Safe,Safe,PASS
4,Cricket Stadium Before Match,28000,88,520,18,Cloudy,0.5,Normal,2,2,...,0,20,0,0,0,68,9,Warning,Dangerous,FAIL


In [172]:
validation_data["Result"] = np.where(
    validation_data["Expected_Risk"] ==
    validation_data["Predicted_Risk"],
    "PASS",
    "FAIL"
)

validation_data[
[
    "Scenario",
    "Expected_Risk",
    "Predicted_Risk",
    "Result"
]
]



,Scenario,Expected_Risk,Predicted_Risk,Result
0,Shopping Mall - Normal Weekend,Safe,Safe,PASS
1,Airport Check-in Rush Hour,Warning,Dangerous,FAIL
2,Railway Station Festival Rush,Dangerous,Dangerous,PASS
3,Corporate Event,Safe,Safe,PASS
4,Cricket Stadium Before Match,Warning,Dangerous,FAIL
5,College Festival Peak Hour,Warning,Dangerous,FAIL
6,Metro Station Evening Rush,Dangerous,Dangerous,PASS
7,Shopping Mall Weekday,Safe,Safe,PASS
8,Music Concert During Rain,Dangerous,Dangerous,PASS
9,Hospital Visitor Hours,Safe,Safe,PASS


In [173]:
total = len(validation_data)

passed = (
    validation_data["Result"] == "PASS"
).sum()

failed = (
    validation_data["Result"] == "FAIL"
).sum()

accuracy = (passed / total) * 100

print("Total Test Cases :", total)
print("Passed :", passed)
print("Failed :", failed)
print("Scenario Validation Accuracy :", round(accuracy,2), "%")

Total Test Cases : 20
Passed : 16
Failed : 4
Scenario Validation Accuracy : 80.0 %


# CONGESTION MODEL VALIDATION

In [174]:
import pickle
import pandas as pd

congestion_model = pickle.load(
    open("../notebooks/congestion_model.pkl", "rb")
)

congestion_scaler = pickle.load(
    open("../notebooks/congestion_scaler.pkl", "rb")
)

congestion_reverse_mapping = pickle.load(
    open("../notebooks/congestion_label_encoder.pkl", "rb")
)

validation = pd.read_csv("congestion_validation.csv")

In [175]:
validation["gate_efficiency"] = validation["gate_efficiency"] / 100


In [176]:
congestion_x = validation.drop(
    columns=[
        "Scenario",
        "Expected_Congestion"
    ]
)

congestion_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather,weather_severity,visibility_km,road_traffic_index,parking_occupancy_pct,crowd_density_per_sqm,crowd_behavior,camera_coverage_pct,volunteer_gap,medical_incidents,security_incidents
0,1800,42,55,52,18,2,6,0.92,Sunny,0.0,10,22,38,1.2,Calm,98,2,0,0
1,2500,48,80,76,35,3,8,0.95,Cloudy,0.5,9,28,45,1.5,Calm,99,3,0,0
2,1400,52,45,42,40,5,4,0.90,Sunny,0.0,10,20,55,1.8,Calm,100,4,1,0
3,4200,68,130,120,120,8,7,0.86,Sunny,0.0,10,42,70,2.6,Normal,97,8,0,0
4,8500,80,210,175,290,11,6,0.78,Cloudy,0.5,8,60,82,3.8,Normal,95,14,1,1


In [177]:
congestion_x = pd.get_dummies(
    congestion_x,
    columns=[
        "weather",
        "crowd_behavior"
    ]
)

congestion_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather_severity,visibility_km,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,1800,42,55,52,18,2,6,0.92,0.0,10,...,False,False,False,False,False,True,False,True,False,False
1,2500,48,80,76,35,3,8,0.95,0.5,9,...,True,False,False,False,False,False,False,True,False,False
2,1400,52,45,42,40,5,4,0.90,0.0,10,...,False,False,False,False,False,True,False,True,False,False
3,4200,68,130,120,120,8,7,0.86,0.0,10,...,False,False,False,False,False,True,False,False,True,False
4,8500,80,210,175,290,11,6,0.78,0.5,8,...,True,False,False,False,False,False,False,False,True,False


In [178]:
training_columns = list(
    congestion_model.feature_names_in_
)

congestion_x = congestion_x.reindex(
    columns=training_columns,
    fill_value=0
)

congestion_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather_severity,visibility_km,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,1800,42,55,52,18,2,6,0.92,0.0,10,...,False,False,False,False,False,True,False,True,False,False
1,2500,48,80,76,35,3,8,0.95,0.5,9,...,True,False,False,False,False,False,False,True,False,False
2,1400,52,45,42,40,5,4,0.90,0.0,10,...,False,False,False,False,False,True,False,True,False,False
3,4200,68,130,120,120,8,7,0.86,0.0,10,...,False,False,False,False,False,True,False,False,True,False
4,8500,80,210,175,290,11,6,0.78,0.5,8,...,True,False,False,False,False,False,False,False,True,False


In [179]:
numeric_cols = list(congestion_scaler.feature_names_in_)

congestion_x[numeric_cols] = congestion_scaler.transform(
    congestion_x[numeric_cols]
)

congestion_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather_severity,visibility_km,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,-0.540248,-1.721837,-0.416746,-0.464857,-0.144725,-0.987548,-0.322713,0.418561,-0.906447,0.949075,...,False,False,False,False,False,True,False,True,False,False
1,-0.488161,-1.402254,-0.369420,-0.408382,-0.121376,-0.659214,0.103733,0.854864,-0.419660,0.495327,...,True,False,False,False,False,False,False,True,False,False
2,-0.570012,-1.189200,-0.435677,-0.488388,-0.114509,-0.002548,-0.749159,0.127692,-0.906447,0.949075,...,False,False,False,False,False,True,False,True,False,False
3,-0.361663,-0.336980,-0.274768,-0.304844,-0.004632,0.982452,-0.109490,-0.454046,-0.906447,0.949075,...,False,False,False,False,False,True,False,False,True,False
4,-0.041699,0.302184,-0.123324,-0.175423,0.228857,1.967452,-0.322713,-1.617520,-0.419660,0.041579,...,True,False,False,False,False,False,False,False,True,False


In [180]:
congestion_prediction = congestion_model.predict(congestion_x)

congestion_prediction = congestion_reverse_mapping.inverse_transform(
    congestion_prediction
)

validation["Predicted_Congestion"] = congestion_prediction

validation.head()

,Scenario,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather,...,road_traffic_index,parking_occupancy_pct,crowd_density_per_sqm,crowd_behavior,camera_coverage_pct,volunteer_gap,medical_incidents,security_incidents,Expected_Congestion,Predicted_Congestion
0,Shopping Mall - Weekday Morning,1800,42,55,52,18,2,6,0.92,Sunny,...,22,38,1.2,Calm,98,2,0,0,Low,Low
1,Corporate Office Entry,2500,48,80,76,35,3,8,0.95,Cloudy,...,28,45,1.5,Calm,99,3,0,0,Low,Low
2,Hospital Visiting Hours,1400,52,45,42,40,5,4,0.90,Sunny,...,20,55,1.8,Calm,100,4,1,0,Low,Low
3,University Convocation,4200,68,130,120,120,8,7,0.86,Sunny,...,42,70,2.6,Normal,97,8,0,0,Medium,Low
4,Temple Festival,8500,80,210,175,290,11,6,0.78,Cloudy,...,60,82,3.8,Normal,95,14,1,1,Medium,Medium


In [181]:
idx = list(congestion_scaler.feature_names_in_).index("gate_efficiency")

print("Mean :", congestion_scaler.mean_[idx])
print("Scale:", congestion_scaler.scale_[idx])

Mean : 0.8912199673021307
Scale: 0.06875955006036298


In [182]:
validation["Result"] = validation.apply(
    lambda row: "PASS"
    if row["Expected_Congestion"] == row["Predicted_Congestion"]
    else "FAIL",
    axis=1
)

validation[
    [
        "Scenario",
        "Expected_Congestion",
        "Predicted_Congestion",
        "Result"
    ]
]

,Scenario,Expected_Congestion,Predicted_Congestion,Result
0,Shopping Mall - Weekday Morning,Low,Low,PASS
1,Corporate Office Entry,Low,Low,PASS
2,Hospital Visiting Hours,Low,Low,PASS
3,University Convocation,Medium,Low,FAIL
4,Temple Festival,Medium,Medium,PASS
5,Airport Check-in Rush,Medium,Medium,PASS
6,Bus Terminal Morning,Medium,Medium,PASS
7,Shopping Mall Weekend,Medium,Low,FAIL
8,Metro Evening Peak,High,High,PASS
9,Railway Festival Rush,High,High,PASS


In [183]:
total = len(validation)

passed = (validation["Result"] == "PASS").sum()

failed = (validation["Result"] == "FAIL").sum()

accuracy = (passed / total) * 100

print("Total Test Cases :", total)
print("Passed :", passed)
print("Failed :", failed)
print("Scenario Validation Accuracy :", round(accuracy, 2), "%")

Total Test Cases : 20
Passed : 15
Failed : 5
Scenario Validation Accuracy : 75.0 %


Waiting Time Validation

In [184]:
import pickle
import pandas as pd

waiting_time_model = pickle.load(
    open("../notebooks/waiting_time_model.pkl", "rb")
)

waiting_time_scaler = pickle.load(
    open("../notebooks/waiting_time_scaler.pkl", "rb")
)

validation = pd.read_csv("waiting_time_validation.csv")

In [185]:
waiting_x = validation.drop(
    columns=[
        "Scenario",
        "Expected_Waiting_Time"
    ]
)

waiting_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,entry_gates,gate_width_m,gate_efficiency,security_check_time_sec,metal_detector_delay_sec,...,device_failure_pct,weather,weather_severity,visibility_km,crowd_behavior,volunteers_assigned,required_volunteers,is_peak_hour,medical_incidents,security_incidents
0,58359.0,89.7,642.9,642.9,0.0,21,2.42,0.788679,16.2,6.5,...,5.0,Rain,2.0,6.53,Normal,668.0,795.0,0,0,0
1,19835.0,65.0,512.1,512.1,6.0,15,1.84,0.908681,18.7,8.8,...,1.7,Sunny,0.0,7.68,Normal,312.0,262.0,1,0,0
2,49502.0,99.1,435.8,435.8,0.0,19,2.42,0.878461,27.4,14.1,...,4.6,Extreme Heat,1.2,10.00,Aggressive,601.0,684.0,0,0,0
3,11352.0,65.7,457.5,457.5,22.0,11,1.53,0.913890,28.9,12.4,...,4.3,Cloudy,0.5,9.50,Normal,138.0,151.0,1,0,0
4,17007.0,77.4,632.3,632.3,6.0,13,1.83,0.831627,19.0,8.5,...,5.2,Fog,1.5,4.12,Normal,249.0,229.0,1,0,0


In [186]:
waiting_x = pd.get_dummies(
    waiting_x,
    columns=[
        "weather",
        "crowd_behavior"
    ]
)

waiting_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,entry_gates,gate_width_m,gate_efficiency,security_check_time_sec,metal_detector_delay_sec,...,security_incidents,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal
0,58359.0,89.7,642.9,642.9,0.0,21,2.42,0.788679,16.2,6.5,...,0,False,False,False,True,False,False,False,False,True
1,19835.0,65.0,512.1,512.1,6.0,15,1.84,0.908681,18.7,8.8,...,0,False,False,False,False,False,True,False,False,True
2,49502.0,99.1,435.8,435.8,0.0,19,2.42,0.878461,27.4,14.1,...,0,False,True,False,False,False,False,True,False,False
3,11352.0,65.7,457.5,457.5,22.0,11,1.53,0.913890,28.9,12.4,...,0,True,False,False,False,False,False,False,False,True
4,17007.0,77.4,632.3,632.3,6.0,13,1.83,0.831627,19.0,8.5,...,0,False,False,True,False,False,False,False,False,True


In [187]:
training_columns = list(
    waiting_time_model.feature_names_in_
)

waiting_x = waiting_x.reindex(
    columns=training_columns,
    fill_value=0
)

waiting_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,entry_gates,gate_width_m,gate_efficiency,security_check_time_sec,metal_detector_delay_sec,...,base_required_volunteers,is_peak_hour,queue_growth_rate_per_min,queue_reduction_rate_per_min,qr_scan_time_sec,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny
0,58359.0,89.7,642.9,642.9,0.0,21,2.42,0.788679,16.2,6.5,...,0,0,0,0,0,False,False,True,False,False
1,19835.0,65.0,512.1,512.1,6.0,15,1.84,0.908681,18.7,8.8,...,0,1,0,0,0,False,False,False,False,True
2,49502.0,99.1,435.8,435.8,0.0,19,2.42,0.878461,27.4,14.1,...,0,0,0,0,0,True,False,False,False,False
3,11352.0,65.7,457.5,457.5,22.0,11,1.53,0.913890,28.9,12.4,...,0,1,0,0,0,False,False,False,False,False
4,17007.0,77.4,632.3,632.3,6.0,13,1.83,0.831627,19.0,8.5,...,0,1,0,0,0,False,True,False,False,False


In [188]:
numeric_cols = list(
    waiting_time_scaler.feature_names_in_
)

waiting_x[numeric_cols] = waiting_time_scaler.transform(
    waiting_x[numeric_cols]
)

waiting_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,entry_gates,gate_width_m,gate_efficiency,security_check_time_sec,metal_detector_delay_sec,...,base_required_volunteers,is_peak_hour,queue_growth_rate_per_min,queue_reduction_rate_per_min,qr_scan_time_sec,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny
0,3.733103,0.813309,0.719568,0.955635,-0.165853,2.901552,2.974093,-1.510223,-0.372680,-0.710949,...,-0.712946,-1.247034,-0.157336,-0.597303,-5.209243,False,False,True,False,False
1,0.819009,-0.504023,0.466040,0.639886,-0.157413,1.611400,1.344974,0.244196,0.006715,0.092472,...,-0.712946,0.801903,-0.157336,-0.597303,-5.209243,False,False,False,False,True
2,3.063127,1.314641,0.318148,0.455700,-0.165853,2.471501,2.974093,-0.197619,1.327009,1.943834,...,-0.712946,-1.247034,-0.157336,-0.597303,-5.209243,True,False,False,False,False
3,0.177325,-0.466689,0.360209,0.508083,-0.134906,0.751299,0.474239,0.320351,1.554646,1.350001,...,-0.712946,0.801903,-0.157336,-0.597303,-5.209243,False,False,False,False,False
4,0.605089,0.157310,0.699022,0.930046,-0.157413,1.181349,1.316886,-0.882329,0.052243,-0.012322,...,-0.712946,0.801903,-0.157336,-0.597303,-5.209243,False,True,False,False,False


In [189]:
waiting_prediction = waiting_time_model.predict(
    waiting_x
)

validation["Predicted_Waiting_Time"] = waiting_prediction.round(2)

validation.head()

,Scenario,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,entry_gates,gate_width_m,gate_efficiency,security_check_time_sec,...,weather_severity,visibility_km,crowd_behavior,volunteers_assigned,required_volunteers,is_peak_hour,medical_incidents,security_incidents,Expected_Waiting_Time,Predicted_Waiting_Time
0,Religious Festival Peak,58359.0,89.7,642.9,642.9,0.0,21,2.42,0.788679,16.2,...,2.0,6.53,Normal,668.0,795.0,0,0,0,4.4,4.74
1,Stadium Entry Rush,19835.0,65.0,512.1,512.1,6.0,15,1.84,0.908681,18.7,...,0.0,7.68,Normal,312.0,262.0,1,0,0,7.9,5.43
2,Concert Main Gate,49502.0,99.1,435.8,435.8,0.0,19,2.42,0.878461,27.4,...,1.2,10.00,Aggressive,601.0,684.0,0,0,0,2.9,3.56
3,Metro Evening Peak,11352.0,65.7,457.5,457.5,22.0,11,1.53,0.913890,28.9,...,0.5,9.50,Normal,138.0,151.0,1,0,0,2.6,6.26
4,Airport Holiday Rush,17007.0,77.4,632.3,632.3,6.0,13,1.83,0.831627,19.0,...,1.5,4.12,Normal,249.0,229.0,1,0,0,6.8,7.10


In [190]:
validation["Difference"] = abs(
    validation["Expected_Waiting_Time"] -
    validation["Predicted_Waiting_Time"]
)

In [191]:
validation["Waiting_Result"] = validation["Difference"].apply(
    lambda x: "PASS" if x <= 3 else "FAIL"
)

In [192]:
validation[
[
    "Scenario",
    "Expected_Waiting_Time",
    "Predicted_Waiting_Time",
    "Difference",
    "Waiting_Result"
]
]

,Scenario,Expected_Waiting_Time,Predicted_Waiting_Time,Difference,Waiting_Result
0,Religious Festival Peak,4.4,4.74,0.34,PASS
1,Stadium Entry Rush,7.9,5.43,2.47,PASS
2,Concert Main Gate,2.9,3.56,0.66,PASS
3,Metro Evening Peak,2.6,6.26,3.66,FAIL
4,Airport Holiday Rush,6.8,7.10,0.30,PASS
5,Shopping Mall Weekend,6.7,3.90,2.80,PASS
6,University Festival,9.5,7.70,1.80,PASS
7,Railway Departure,6.8,4.71,2.09,PASS
8,Corporate Expo,6.8,5.45,1.35,PASS
9,City Marathon,9.7,7.54,2.16,PASS


In [193]:
total = len(validation)

passed = (
    validation["Waiting_Result"] == "PASS"
).sum()

failed = total - passed

accuracy = (passed / total) * 100

print("Total Test Cases :", total)
print("Passed :", passed)
print("Failed :", failed)
print("Waiting Time Validation Accuracy :", round(accuracy,2), "%")

Total Test Cases : 20
Passed : 19
Failed : 1
Waiting Time Validation Accuracy : 95.0 %


congestion score prediction




In [194]:
import pickle
import pandas as pd

congestion_score_model = pickle.load(
    open("../notebooks/congestion_score_model.pkl", "rb")
)

congestion_score_scaler = pickle.load(
    open("../notebooks/congestion_score_scaler.pkl", "rb")
)

validation = pd.read_csv(
    "congestion_score_validation.csv"
)

In [195]:
con_x = validation.drop(
    columns=[
        "Scenario",
        "Expected_Congestion_Score"
    ]
)

con_x.head()

,event_type,time_slot,day_type,weather,weather_severity,venue_capacity,is_indoor,venue_size_sqm,entry_gates,emergency_gates,...,fire_alarm,stampede_alert,bomb_threat,volunteers_assigned,required_volunteers,volunteer_gap,reserve_volunteers,congestion_level,risk_score,risk_level
0,Political Rally,Afternoon,Festival Day,Sunny,0.0,147022.0,0,59731.0,31,9,...,0,0,0,1984.0,2194.0,210.0,273.0,Low,29.9,Dangerous
1,Religious Gathering,Evening,Festival Day,Cloudy,0.5,138283.0,0,46591.0,31,8,...,0,0,0,1534.0,1993.0,459.0,376.0,Medium,47.4,Dangerous
2,Religious Gathering,Morning,Weekend,Cloudy,0.5,129341.0,1,44178.0,31,10,...,0,0,0,1221.0,1706.0,485.0,272.0,Low,26.2,Warning
3,Religious Gathering,Evening,Weekend,Cloudy,0.5,114387.0,0,38017.0,29,6,...,0,0,0,1404.0,1706.0,302.0,205.0,High,65.2,Dangerous
4,Religious Gathering,Afternoon,Weekend,Fog,1.5,71800.0,1,25793.0,23,9,...,0,0,0,581.0,875.0,294.0,204.0,Medium,43.7,Dangerous


In [196]:
con_x = pd.get_dummies(
    con_x,
    columns=[
        "weather",
        "crowd_behavior"
    ]
)

con_x.head()

,event_type,time_slot,day_type,weather_severity,venue_capacity,is_indoor,venue_size_sqm,entry_gates,emergency_gates,gate_width_m,...,weather_Cloudy,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Aggressive,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,Political Rally,Afternoon,Festival Day,0.0,147022.0,0,59731.0,31,9,4.19,...,False,False,False,False,False,True,False,False,False,True
1,Religious Gathering,Evening,Festival Day,0.5,138283.0,0,46591.0,31,8,4.02,...,True,False,False,False,False,False,False,False,True,False
2,Religious Gathering,Morning,Weekend,0.5,129341.0,1,44178.0,31,10,3.71,...,True,False,False,False,False,False,False,False,True,False
3,Religious Gathering,Evening,Weekend,0.5,114387.0,0,38017.0,29,6,3.35,...,True,False,False,False,False,False,False,False,True,False
4,Religious Gathering,Afternoon,Weekend,1.5,71800.0,1,25793.0,23,9,2.44,...,False,False,True,False,False,False,False,False,True,False


In [197]:
training_columns = list(
    congestion_score_model.feature_names_in_
)

con_x = con_x.reindex(
    columns=training_columns,
    fill_value=0
)

con_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather_severity,visibility_km,...,security_incidents,is_peak_hour,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,158784.0,108.0,4505.3,4505.3,2.0,1.6,31,0.962659,0.0,9.80,...,0,1,False,False,False,False,True,False,False,True
1,144352.0,104.4,7951.8,5480.8,6000.0,8.0,31,0.909425,0.5,9.04,...,0,1,False,False,False,False,False,False,True,False
2,124204.0,96.0,5212.1,5000.3,10.0,6.0,31,0.886098,0.5,8.95,...,0,1,False,False,False,False,False,False,True,False
3,123538.0,108.0,10047.1,4379.0,6000.0,9.5,29,0.932989,0.5,9.41,...,1,1,False,False,False,False,False,False,True,False
4,64210.0,89.4,3223.4,2323.0,6000.0,9.8,23,0.879439,1.5,3.44,...,0,1,False,True,False,False,False,False,True,False


In [198]:
numeric_cols = list(
    congestion_score_scaler.feature_names_in_
)

con_x[numeric_cols] = congestion_score_scaler.transform(
    con_x[numeric_cols]
)

con_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,avg_waiting_time_min,entry_gates,gate_efficiency,weather_severity,visibility_km,...,security_incidents,is_peak_hour,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,11.140980,1.793568,8.007880,10.014301,-0.166700,-1.118881,5.007860,1.038974,-0.906447,0.858325,...,-0.207684,0.802281,False,False,False,False,True,False,False,True
1,10.067090,1.601819,14.532265,12.309771,8.071340,0.982452,5.007860,0.264769,-0.419660,0.513477,...,-0.207684,0.802281,False,False,False,False,False,False,True,False
2,8.567872,1.154403,9.345885,11.179096,-0.155713,0.325785,5.007860,-0.074486,-0.419660,0.472639,...,-0.207684,0.802281,False,False,False,False,False,False,True,False
3,8.518314,1.793568,18.498766,9.717102,8.071340,1.474952,4.581414,0.607469,-0.419660,0.681363,...,1.824453,0.802281,False,False,False,False,False,False,True,False
4,4.103699,0.802863,5.581183,4.879083,8.071340,1.573452,3.302077,-0.171338,0.553916,-2.027512,...,-0.207684,0.802281,False,True,False,False,False,False,True,False


In [199]:
prediction = congestion_score_model.predict(
    con_x
)

validation["Predicted_Congestion_Score"] = prediction.round(2)

validation.head()

,Scenario,event_type,time_slot,day_type,weather,weather_severity,venue_capacity,is_indoor,venue_size_sqm,entry_gates,...,bomb_threat,volunteers_assigned,required_volunteers,volunteer_gap,reserve_volunteers,Expected_Congestion_Score,congestion_level,risk_score,risk_level,Predicted_Congestion_Score
0,Kumbh Peak,Political Rally,Afternoon,Festival Day,Sunny,0.0,147022.0,0,59731.0,31,...,0,1984.0,2194.0,210.0,273.0,27.3,Low,29.9,Dangerous,30.62
1,Religious Festival,Religious Gathering,Evening,Festival Day,Cloudy,0.5,138283.0,0,46591.0,31,...,0,1534.0,1993.0,459.0,376.0,57.6,Medium,47.4,Dangerous,60.82
2,Stadium Entry,Religious Gathering,Morning,Weekend,Cloudy,0.5,129341.0,1,44178.0,31,...,0,1221.0,1706.0,485.0,272.0,32.3,Low,26.2,Warning,33.67
3,Concert Rush,Religious Gathering,Evening,Weekend,Cloudy,0.5,114387.0,0,38017.0,29,...,0,1404.0,1706.0,302.0,205.0,67.0,High,65.2,Dangerous,62.96
4,Metro Peak,Religious Gathering,Afternoon,Weekend,Fog,1.5,71800.0,1,25793.0,23,...,0,581.0,875.0,294.0,204.0,61.1,Medium,43.7,Dangerous,60.63


In [200]:
validation["Difference"] = abs(
    validation["Expected_Congestion_Score"] -
    validation["Predicted_Congestion_Score"]
)

In [201]:
validation["Congestion_Result"] = validation["Difference"].apply(
    lambda x: "PASS" if x <= 5 else "FAIL"
)

In [202]:
validation[
[
    "Scenario",
    "Expected_Congestion_Score",
    "Predicted_Congestion_Score",
    "Difference",
    "Congestion_Result"
]
]

,Scenario,Expected_Congestion_Score,Predicted_Congestion_Score,Difference,Congestion_Result
0,Kumbh Peak,27.3,30.62,3.32,PASS
1,Religious Festival,57.6,60.82,3.22,PASS
2,Stadium Entry,32.3,33.67,1.37,PASS
3,Concert Rush,67.0,62.96,4.04,PASS
4,Metro Peak,61.1,60.63,0.47,PASS
5,Airport Rush,60.9,60.30,0.60,PASS
6,Mall Weekend,58.5,59.30,0.80,PASS
7,Railway Platform,58.3,59.05,0.75,PASS
8,Storm Event,75.9,66.55,9.35,FAIL
9,Heavy Rain,33.1,30.25,2.85,PASS


In [203]:
total = len(validation)

passed = (
    validation["Congestion_Result"] == "PASS"
).sum()

failed = total - passed

accuracy = (passed / total) * 100

print("Total Test Cases :", total)
print("Passed :", passed)
print("Failed :", failed)
print("Congestion Score Validation Accuracy :", round(accuracy,2), "%")

Total Test Cases : 20
Passed : 18
Failed : 2
Congestion Score Validation Accuracy : 90.0 %


In [204]:
import pickle
import pandas as pd

volunteer_model = pickle.load(
    open("../notebooks/volunteer_model.pkl", "rb")
)

volunteer_scaler = pickle.load(
    open("../notebooks/volunteer_scaler.pkl", "rb")
)

validation = pd.read_csv(
    "volunteer_validation.csv"
)

In [205]:
volunteer_x = validation.drop(
    columns=[
        "Scenario",
        "Expected_Volunteers"
    ]
)

volunteer_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,weather,weather_severity,medical_incidents,security_incidents,entry_gates,gate_efficiency,is_peak_hour,crowd_behavior,venue_capacity
0,850,68,45,42,40,Sunny,1,0,0,4,92,0,Normal,1500
1,1200,72,55,52,55,Cloudy,2,0,0,5,90,1,Normal,2000
2,1800,75,70,65,80,Sunny,1,1,0,6,88,1,Normal,3000
3,2500,78,90,84,120,Cloudy,2,0,1,8,87,1,Normal,5000
4,3200,82,110,98,160,Rain,3,1,1,10,84,1,Excited,6000


In [206]:
volunteer_x = pd.get_dummies(
    volunteer_x,
    columns=[
        "weather",
        "crowd_behavior"
    ],
    drop_first=True
)

training_columns = volunteer_model.feature_names_in_

volunteer_x = volunteer_x.reindex(
    columns=training_columns,
    fill_value=0
)


training_columns = volunteer_model.feature_names_in_

volunteer_x = volunteer_x.reindex(
    columns=training_columns,
    fill_value=0
)

volunteer_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,weather_severity,medical_incidents,security_incidents,entry_gates,gate_efficiency,is_peak_hour,venue_capacity,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,850,68,45,42,40,1,0,0,4,92,0,1500,0,0,False,0,True,0,True,False
1,1200,72,55,52,55,2,0,0,5,90,1,2000,0,0,False,0,False,0,True,False
2,1800,75,70,65,80,1,1,0,6,88,1,3000,0,0,False,0,True,0,True,False
3,2500,78,90,84,120,2,0,1,8,87,1,5000,0,0,False,0,False,0,True,False
4,3200,82,110,98,160,3,1,1,10,84,1,6000,0,0,True,0,False,0,False,False


In [207]:
numeric_cols = [
    "crowd_count",
    "occupancy_pct",
    "arrival_rate_per_min",
    "entry_rate_per_min",
    "queue_length",
    "weather_severity",
    "medical_incidents",
    "security_incidents",
    "entry_gates",
    "gate_efficiency",
    "is_peak_hour",      # <-- Add this
    "venue_capacity"
]
volunteer_x[numeric_cols] = volunteer_scaler.transform(
    volunteer_x[numeric_cols]
)

volunteer_x.head()

,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,weather_severity,medical_incidents,security_incidents,entry_gates,gate_efficiency,is_peak_hour,venue_capacity,weather_Extreme Heat,weather_Fog,weather_Rain,weather_Storm,weather_Sunny,crowd_behavior_Calm,crowd_behavior_Normal,crowd_behavior_Panic
0,-0.610938,-0.336980,-0.435677,-0.488388,-0.114509,0.067128,-0.236235,-0.207684,-0.749159,1325.034558,-1.246445,-0.648258,0,0,False,0,True,0,True,False
1,-0.584894,-0.123925,-0.416746,-0.464857,-0.093907,1.040703,-0.236235,-0.207684,-0.535936,1295.947690,0.802281,-0.615017,0,0,False,0,False,0,True,False
2,-0.540248,0.035866,-0.388350,-0.434266,-0.059570,0.067128,1.244865,-0.207684,-0.322713,1266.860821,0.802281,-0.548535,0,0,False,0,True,0,True,False
3,-0.488161,0.195657,-0.350489,-0.389557,-0.004632,1.040703,-0.236235,1.824453,0.103733,1252.317387,0.802281,-0.415572,0,0,False,0,False,0,True,False
4,-0.436073,0.408712,-0.312629,-0.356613,0.050307,2.014278,1.244865,1.824453,0.530179,1208.687084,0.802281,-0.349090,0,0,True,0,False,0,False,False


In [208]:
prediction = volunteer_model.predict(
    volunteer_x
)

validation["Predicted_Volunteers"] = prediction.round().astype(int)

validation.head()

,Scenario,crowd_count,occupancy_pct,arrival_rate_per_min,entry_rate_per_min,queue_length,weather,weather_severity,medical_incidents,security_incidents,entry_gates,gate_efficiency,is_peak_hour,crowd_behavior,venue_capacity,Expected_Volunteers,Predicted_Volunteers
0,University Tech Fest,850,68,45,42,40,Sunny,1,0,0,4,92,0,Normal,1500,8,11
1,College Convocation,1200,72,55,52,55,Cloudy,2,0,0,5,90,1,Normal,2000,12,15
2,Shopping Mall Weekend,1800,75,70,65,80,Sunny,1,1,0,6,88,1,Normal,3000,18,22
3,Industrial Expo,2500,78,90,84,120,Cloudy,2,0,1,8,87,1,Normal,5000,25,29
4,Navratri Garba Night,3200,82,110,98,160,Rain,3,1,1,10,84,1,Excited,6000,35,35


In [209]:
validation["Difference"] = abs(
    validation["Expected_Volunteers"] -
    validation["Predicted_Volunteers"]
)

In [210]:
validation["Percent_Error"] = (
    abs(
        validation["Expected_Volunteers"] -
        validation["Predicted_Volunteers"]
    )
    /
    validation["Expected_Volunteers"]
) * 100

validation["Volunteer_Result"] = validation["Percent_Error"].apply(
    lambda x: "PASS" if x <= 15 else "FAIL"
)

In [211]:
validation[
[
    "Scenario",
    "Expected_Volunteers",
    "Predicted_Volunteers",
    "Difference",
    "Volunteer_Result"
]
]

,Scenario,Expected_Volunteers,Predicted_Volunteers,Difference,Volunteer_Result
0,University Tech Fest,8,11,3,FAIL
1,College Convocation,12,15,3,FAIL
2,Shopping Mall Weekend,18,22,4,FAIL
3,Industrial Expo,25,29,4,FAIL
4,Navratri Garba Night,35,35,0,PASS
5,Music Concert,45,43,2,PASS
6,Metro Peak Hour,52,58,6,PASS
7,Temple Festival,65,77,12,FAIL
8,Railway Holiday Rush,80,91,11,PASS
9,Airport Departure Rush,92,103,11,PASS


In [212]:
total = len(validation)

passed = (
    validation["Volunteer_Result"] == "PASS"
).sum()

failed = total - passed

accuracy = (passed / total) * 100

print("Total Test Cases :", total)
print("Passed :", passed)
print("Failed :", failed)
print("Volunteer Validation Accuracy :", round(accuracy, 2), "%")

Total Test Cases : 20
Passed : 12
Failed : 8
Volunteer Validation Accuracy : 60.0 %


cluster model validation


In [213]:
import pandas as pd
import pickle

validation = pd.read_csv("cluster_validation.csv")

with open("../notebooks/crowd_cluster_model.pkl", "rb") as file:
    cluster_model = pickle.load(file)

with open("../notebooks/cluster_scaler.pkl", "rb") as file:
    cluster_scaler = pickle.load(file)

In [214]:
cluster_x = validation[[
    "crowd_count",
    "occupancy_pct",
    "queue_length",
    "avg_waiting_time_min",
    "congestion_score",
    "risk_score",
    "crowd_density_per_sqm",
    "volunteer_gap"
]]

In [215]:
cluster_x = cluster_scaler.transform(cluster_x)

In [216]:
cluster_prediction = cluster_model.predict(cluster_x)

In [217]:
cluster_map = {
    0: "Low Crowd",
    2: "Medium Crowd",
    3: "High Crowd",
    1: "Critical Crowd"
}

validation["Predicted_Cluster"] = [
    cluster_map[i] for i in cluster_prediction
]

In [218]:
validation["Cluster_Result"] = validation.apply(
    lambda row:
    "PASS"
    if row["Expected_Cluster"] == row["Predicted_Cluster"]
    else "FAIL",
    axis=1
)

In [219]:
print(validation[[
    "Scenario",
    "Expected_Cluster",
    "Predicted_Cluster",
    "Cluster_Result"
]])

                          Scenario Expected_Cluster Predicted_Cluster  \
0             Small Community Fair        Low Crowd         Low Crowd   
1                School Annual Day        Low Crowd         Low Crowd   
2                College Tech Fest     Medium Crowd      Medium Crowd   
3            Shopping Mall Weekend     Medium Crowd      Medium Crowd   
4             Temple Weekend Crowd     Medium Crowd        High Crowd   
5                  Metro Peak Hour     Medium Crowd        High Crowd   
6                  Industrial Expo       High Crowd        High Crowd   
7                    Music Concert       High Crowd        High Crowd   
8             Airport Holiday Rush       High Crowd        High Crowd   
9                IPL Cricket Match       High Crowd        High Crowd   
10        International Trade Expo       High Crowd        High Crowd   
11                 Political Rally   Critical Crowd    Critical Crowd   
12             Garba Festival Exit   Critical Crowd

In [220]:
passed = (
    validation["Cluster_Result"] == "PASS"
).sum()

failed = (
    validation["Cluster_Result"] == "FAIL"
).sum()

accuracy = (
    passed / len(validation)
) * 100

print()

print("Total Test Cases :", len(validation))
print("Passed :", passed)
print("Failed :", failed)
print("Cluster Validation Accuracy :", round(accuracy,2), "%")


Total Test Cases : 20
Passed : 18
Failed : 2
Cluster Validation Accuracy : 90.0 %
